## **Load Bronze Orders**

In [0]:

bronze_orders_df = spark.read.table("retail_project.bronze.orders")

display(bronze_orders_df)

## Retail Orders - Silver layer cleaning

## Fix Date Fomat

In [0]:
from pyspark.sql.functions import *

silver_bronze_df = bronze_orders_df.withColumn(
    "order_date", coalesce (try_to_date(col("order_date"), "yyyy-MM-dd"),
                            try_to_date(col("order_date"), "dd-MM-yyyy"),
                            try_to_date(col("order_date"), "yyyy/MM/dd"),
                            try_to_date(col("order_date"), "dd/MM/yyyy")
                            ))
silver_bronze_df.display()

### Clean Category Value

In [0]:
silver_bronze_df = silver_bronze_df.withColumn("category" , upper(trim(col("category"))))

silver_bronze_df = silver_bronze_df.withColumn(
                                "category", 
                                 when(col("category") == "ELECTRONICS" , "ELECTRONIC")
                                 .when(col("category") =="FASHON" , "FASHION")
                                 .when(col("category") == "HOME & LIVING" , "HOME AND LIVING")
                                 .otherwise(col("category")))

silver_bronze_df.select("category").distinct().display()                                               


### Remove Duplicate Orders

In [0]:
silver_bronze_df.select(count("order_id"),countDistinct("order_id")).show()

In [0]:
silver_bronze_df = silver_bronze_df.dropDuplicates(["order_id"])
silver_bronze_df.select(count("order_id"),countDistinct("order_id")).show()

Handel Null values in discount

In [0]:
silver_bronze_df = silver_bronze_df.withColumn('discount',
                                        when(col("discount").isNull(),'0')
                                        .when(col("discount")=='ten','0.1')
                                        .otherwise(col("discount"))
                                        )

In [0]:
silver_bronze_df.display()

Convert 5% into decimal

In [0]:
silver_bronze_df = silver_bronze_df.withColumn('discount', 
                                               when(col("discount").like("5%"),
                                                    (regexp_replace("discount", "5%", "0.05"))
                                                ).when(col("discount").like("10%"),
                                                       (regexp_replace("discount","10%","0.01"))
                                                ).otherwise(col("discount"))
                                                    )
silver_bronze_df.display()


## Remove Negative Quantity

In [0]:
silver_bronze_df = silver_bronze_df.filter(col("quantity")> 0)
silver_bronze_df.display()

### Unit Price Into Decimal

In [0]:
from pyspark.sql.types import *
silver_bronze_df = silver_bronze_df.withColumn("unit_price" , col("unit_price").try_cast(DecimalType(10,2)))

silver_bronze_df.display()

### Calculate Total Amount

In [0]:
silver_bronze_df = silver_bronze_df.withColumn("total_amount" , col("quantity").cast("int") * col("unit_price") * (1- col("discount").cast("double")))
silver_bronze_df.display()    

In [0]:
silver_bronze_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_customers")

## Retail Customer - Silver layer Cleaning

In [0]:
silver_customer = spark.read.table("retail_project.bronze.customer")

silver_customer.display()

### Trim Spaces

In [0]:
from pyspark.sql.functions import *

silver_customer = silver_customer.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    trim(col("IngestTime")).alias("IngestTime")
)

silver_customer.display()

### Fix Date format

In [0]:
silver_customer = silver_customer.withColumn("date_of_birth", 
                                      coalesce(try_to_date(col("date_of_birth"), "yyyy-MM-dd"),
                                               try_to_date(col("date_of_birth"), "dd-MM-yyyy"),
                                               try_to_date(col("date_of_birth"), "dd/MM/yyyy"),
                                               try_to_date(col("date_of_birth"), "yyyy/MM/dd"))
                                      )
silver_customer = silver_customer.withColumn("registration_date", 
                                      coalesce(try_to_date(col("registration_date"), "yyyy-MM-dd"),
                                               try_to_date(col("registration_date"), "dd-MM-yyyy"),
                                               try_to_date(col("registration_date"), "dd/MM/yyyy"),
                                               try_to_date(col("registration_date"), "yyyy/MM/dd"))
                                      )                                      
silver_customer.display() 

## Remove Duplicate Customers

In [0]:
from pyspark.sql.window import Window 


W = Window.partitionBy("customer_id").orderBy(col("IngestTime").desc())

silver_customer = silver_customer.withColumn("row" , row_number().over(W))

silver_customer = silver_customer.filter(col("row") =='1').drop("row")
silver_customer.display()

### Fix Status

In [0]:
silver_customer = silver_customer.withColumn("status" , upper(col("status")))

silver_customer = silver_customer.withColumn("status" , when(col("status").isin("ACTIVE" , "INACTIVE"), col("status"))
                                   .otherwise("UNKNOWN"))

silver_customer.display()

Convert Loyalty Points into Int

In [0]:
silver_customer = silver_customer.withColumn("loyalty_points" , regexp_replace(col("loyalty_points") ,"[^0-9]", ""))

silver_customer = silver_customer.withColumn("loyalty_points" , col("loyalty_points").try_cast("int"))
silver_customer.display()

### Validate Emails

In [0]:
silver_customer = silver_customer.withColumn("email_valid" , col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"))

silver_customer = silver_customer.filter(col("email_valid") == True)

silver_customer.display()

In [0]:
silver_customer = silver_customer.withColumn("gender" , upper(col("gender")))

silver_customer = silver_customer.withColumn("gender" , when(col("gender") == "MALE", "M").when(col("gender") == "FEMALE", "F").otherwise("U"))

silver_customer.display()

In [0]:
silver_customer.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_customer")